# 卵巢癌 Xenium + Flex scRNA-seq 数据预处理

**数据集**
- scRNA-seq：Xenium Flex 平台，`17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5`
- 预标注：`FLEX_Ovarian_Barcode_Cluster_Annotation.csv`
- Xenium：`Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/`

**Cell 执行顺序**
| Cell | 内容 | 输出 |
|---|---|---|
| 1 | 初始化路径、参数、细胞类型映射 | — |
| 2 | 加载 Flex scRNA + 附加 CSV 注释 + QC | `.rds` 缓存 |
| 3 | scRNA UMAP 可视化（聚类 + 细胞类型双图） | `plots/02_annotation/` |
| 4 | 加载 Xenium 基础对象 + 独立聚类 + 空间图 | `.rds` 缓存，`plots/04_spatial/` |
| 4.5 | 基因表达相关性分析（平台交叉验证） | `plots/01_quality_control/` |
| 4b | Seurat 标签转移（TransferData，增强版） | `seurat_label_transfer_ovarian.csv` |
| 4c | 导出结果（cell_groups / GNN 专用 / Seurat 对象）| `results/` |
| 5 | 完成确认 | — |

⚠️ **Python 训练 notebook 不依赖本 notebook 的 `.rds` 输出**，  
但需要 Cell 4b 生成的 `seurat_label_transfer_ovarian.csv` 用于对比实验。

In [ ]:
# ============================================================
# Cell 1: 统一初始化 — 路径 / 参数 / 细胞类型映射
# ============================================================

setwd("/home/ailab/caohao/AdaDiss/")

suppressPackageStartupMessages({
    library(Seurat)
    library(dplyr)
    library(ggplot2)
    library(jsonlite)
    library(Matrix)
    library(viridis)
    library(cowplot)
})

# ── 文件路径（与原始定义完全一致，不修改）─────────────────────
PATHS <- list(
    raw = list(
        flex_h5     = "data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5",
        flex_annot  = "data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv",
        xenium_dir  = "data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/",
        gene_panel  = "data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/gene_panel.json"
    ),
    cache = list(
        scrna_annotated = "data/cache/scrna/scrna_annotated_ovarian.rds",
        xenium_base     = "data/cache/xenium/xenium_base_ovarian.rds"
    ),
    # ── 输出路径（新增，供导出和对比用）────────────────────────
    predictions = list(
        seurat_lt   = "data/cache/xenium/seurat_label_transfer_ovarian.csv",
        main        = "results/predictions/cell_groups.csv",
        full        = "results/predictions/cell_predictions_full.csv",
        seurat_obj  = "results/seurat/xenium_annotated_ovarian.rds"
    ),
    plots = list(
        qc          = "plots/01_quality_control/",
        annotation  = "plots/02_annotation/",
        spatial     = "plots/04_spatial/"
    ),
    reports = list(
        stats = "reports/prediction_stats.txt"
    )
)

# ── 创建所有输出目录 ─────────────────────────────────────────
dirs_to_create <- c(
    "data/cache/scrna", "data/cache/xenium",
    "results/predictions", "results/seurat",
    "plots/01_quality_control", "plots/02_annotation", "plots/04_spatial",
    "reports", "figures"
)
for (d in dirs_to_create) dir.create(d, recursive = TRUE, showWarnings = FALSE)

# ── 分析参数 ────────────────────────────────────────────────
params <- list(
    # Flex QC
    flex_min_counts  = 200,
    flex_max_counts  = 10000,
    flex_max_mt      = 25,
    # 标签转移
    transfer_dims    = 1:30,
    transfer_k       = 50,
    conf_threshold   = 0.6,     # 低置信度过滤阈值
    seed             = 42
)
set.seed(params$seed)
options(future.globals.maxSize = 1e9)

# ── 细胞类型映射表（点号列名 ↔ 原始名称）─────────────────────
# TransferData 返回的分数列名含点号（R 不允许空格），需要映射回原始名称
cell_type_mapping <- c(
    "Tumor.Associated.Fibroblasts"      = "Tumor Associated Fibroblasts",
    "Endothelial.Cells"                 = "Endothelial Cells",
    "Stromal.Associated.Fibroblasts"    = "Stromal Associated Fibroblasts",
    "T...NK.Cells"                      = "T & NK Cells",
    "Malignant.Cells.Lining.Cyst"       = "Malignant Cells Lining Cyst",
    "Proliferative.Tumor.Cells"         = "Proliferative Tumor Cells",
    "Tumor.Cells"                       = "Tumor Cells",
    "Pericytes"                         = "Pericytes",
    "Granulosa.Cells"                   = "Granulosa Cells",
    "Macrophages"                       = "Macrophages",
    "MT.High..Jun..Fos..Tumor.Cells"    = "MT-High, Jun+/Fos+ Tumor Cells",
    "VEGFA..Tumor.Cells"                = "VEGFA+ Tumor Cells",
    "Smooth.Muscle.Cells"               = "Smooth Muscle Cells",
    "Inflammatory.Tumor.Cells"          = "Inflammatory Tumor Cells",
    "Ciliated.Epithelial.Cells"         = "Ciliated Epithelial Cells",
    "Fallopian.Tube.Epithelium"         = "Fallopian Tube Epithelium"
)
reverse_mapping <- setNames(names(cell_type_mapping), cell_type_mapping)

# ── 缓存加载辅助函数 ─────────────────────────────────────────
load_or_process <- function(cache_file, process_func, force = FALSE) {
    if (!force && file.exists(cache_file)) {
        cat("[Cache HIT]", basename(cache_file), "\n")
        return(readRDS(cache_file))
    }
    cat("[Cache MISS] 处理中...\n")
    obj <- process_func()
    dir.create(dirname(cache_file), recursive = TRUE, showWarnings = FALSE)
    saveRDS(obj, cache_file)
    cat("[Saved]", basename(cache_file), "\n")
    return(obj)
}

# ── 路径检查 ────────────────────────────────────────────────
cat("=== 路径检查 ===\n")
cat("Flex H5   :", file.exists(PATHS$raw$flex_h5),    "\n")
cat("Flex CSV  :", file.exists(PATHS$raw$flex_annot), "\n")
cat("Xenium dir:", dir.exists(PATHS$raw$xenium_dir),  "\n")
cat("Gene panel:", file.exists(PATHS$raw$gene_panel), "\n")
cat("\n细胞类型映射数:", length(cell_type_mapping), "\n")

In [ ]:
# ============================================================
# Cell 2: 加载 Flex scRNA-seq + 附加预标注 + QC
# ============================================================

scrna_obj <- load_or_process(PATHS$cache$scrna_annotated, function() {

    cat("  步骤1: 加载 H5 计数矩阵...\n")
    counts    <- Read10X_h5(PATHS$raw$flex_h5)
    scrna_obj <- CreateSeuratObject(
        counts    = counts,
        project   = "Ovarian_Flex",
        min.cells = 3, min.features = 200
    )
    cat("  原始细胞数:", ncol(scrna_obj), "  基因数:", nrow(scrna_obj), "\n")

    cat("  步骤2: 加载 Flex 预标注 CSV...\n")
    ann_df <- read.csv(PATHS$raw$flex_annot, stringsAsFactors = FALSE)
    cat("  CSV 列名:", paste(colnames(ann_df), collapse = ", "), "\n")
    print(head(ann_df, 3))

    # 自动识别 barcode 列和 cell_type 列
    bc_col <- grep("barcode|cell_id|barcodes",
                   colnames(ann_df), ignore.case = TRUE, value = TRUE)[1]
    ct_col <- grep("cell_type|celltype|annotation|cluster_name|label",
                   colnames(ann_df), ignore.case = TRUE, value = TRUE)[1]

    # 若自动识别失败，使用原始 notebook 中的确定列名
    if (is.na(bc_col)) bc_col <- "Barcode"
    if (is.na(ct_col)) ct_col <- "Cell.Annotation"
    cat("  barcode 列:", bc_col, "  cell_type 列:", ct_col, "\n")

    # 匹配细胞
    cell_types_map <- setNames(ann_df[[ct_col]], ann_df[[bc_col]])
    common_bc      <- intersect(colnames(scrna_obj), names(cell_types_map))
    cat("  有标注细胞:", length(common_bc), "/", ncol(scrna_obj), "\n")
    scrna_obj <- scrna_obj[, common_bc]
    scrna_obj <- AddMetaData(scrna_obj,
                             cell_types_map[colnames(scrna_obj)], "cell_type")
    cat("  注释了", length(unique(scrna_obj$cell_type)), "种细胞类型\n")

    cat("  步骤3: 线粒体 QC...\n")
    scrna_obj[["percent.mt"]] <- PercentageFeatureSet(scrna_obj, pattern = "^MT-")
    n_before <- ncol(scrna_obj)
    scrna_obj <- subset(scrna_obj,
        subset = nCount_RNA   > params$flex_min_counts &
                 nCount_RNA   < params$flex_max_counts &
                 percent.mt   < params$flex_max_mt)
    cat("  QC 前:", n_before, "→ QC 后:", ncol(scrna_obj), "\n")

    cat("  步骤4: 标准化 + 降维（供可视化）...\n")
    scrna_obj <- NormalizeData(scrna_obj, verbose = FALSE) %>%
                 FindVariableFeatures(nfeatures = 3000, verbose = FALSE) %>%
                 ScaleData(verbose = FALSE) %>%
                 RunPCA(npcs = 30, verbose = FALSE) %>%
                 RunUMAP(dims = 1:20, verbose = FALSE) %>%
                 FindNeighbors(dims = 1:20, verbose = FALSE) %>%
                 FindClusters(resolution = 0.5, verbose = FALSE)

    return(scrna_obj)
})

cat("\n=== Flex scRNA-seq 摘要 ===\n")
cat("细胞数:", ncol(scrna_obj), "\n")
cat("基因数:", nrow(scrna_obj), "\n")
cat("\n细胞类型分布（降序）:\n")
print(sort(table(scrna_obj$cell_type), decreasing = TRUE))

In [ ]:
# ============================================================
# Cell 3: scRNA UMAP 双图可视化
# 左：Seurat 无监督聚类（验证聚类是否对应细胞类型）
# 右：Flex 预标注细胞类型（用于论文 Fig 1 scRNA reference）
# ============================================================

p_cluster  <- DimPlot(scrna_obj, reduction = "umap",
                      group.by = "seurat_clusters",
                      label = TRUE, pt.size = 0.2, repel = TRUE) +
              ggtitle("Flex scRNA — Seurat Clusters") +
              theme(plot.title = element_text(hjust = 0.5),
                    legend.text = element_text(size = 7))

p_celltype <- DimPlot(scrna_obj, reduction = "umap",
                      group.by = "cell_type",
                      label = TRUE, pt.size = 0.2, repel = TRUE) +
              ggtitle("Flex scRNA — Cell Type Annotation") +
              theme(plot.title = element_text(hjust = 0.5),
                    legend.text = element_text(size = 7))

p_combined <- plot_grid(p_cluster, p_celltype, ncol = 2)
print(p_combined)

ggsave(file.path(PATHS$plots$annotation, "flex_umap_cluster_vs_celltype.pdf"),
       p_combined, width = 16, height = 7)
# 也保存单张 cell_type 图供论文使用
ggsave("figures/scrna_umap_ovarian.pdf", p_celltype, width = 10, height = 8)

cat("论文图已保存：figures/scrna_umap_ovarian.pdf\n")
cat("双图已保存：", file.path(PATHS$plots$annotation, "flex_umap_cluster_vs_celltype.pdf"), "\n")

In [ ]:
# ============================================================
# Cell 4: 加载 Xenium + 独立聚类 + 空间可视化
# 论文价值：展示空间结构（标注前），与标注后形成对比
# ============================================================

xenium_obj <- load_or_process(PATHS$cache$xenium_base, function() {

    cat("  步骤1: LoadXenium...\n")
    obj <- LoadXenium(PATHS$raw$xenium_dir, fov = "fov",
                      molecule.coordinates = FALSE)
    DefaultAssay(obj) <- "Xenium"

    cat("  步骤2: 过滤空细胞...\n")
    n_before <- ncol(obj)
    obj <- subset(obj, subset = nCount_Xenium > 0)
    cat("  过滤前:", n_before, "→ 过滤后:", ncol(obj), "\n")

    # log 转换计数（备用元数据，便于后续 QC 可视化）
    obj@meta.data$nCount_Xenium_log  <- log1p(obj@meta.data$nCount_Xenium)
    obj@meta.data$nFeature_Xenium_log <- log1p(obj@meta.data$nFeature_Xenium)

    cat("  步骤3: 归一化 + 降维 + 聚类...\n")
    obj <- NormalizeData(obj, verbose = FALSE) %>%
           FindVariableFeatures(verbose = FALSE) %>%
           ScaleData(verbose = FALSE) %>%
           RunPCA(npcs = 50, verbose = FALSE) %>%
           RunUMAP(dims = 1:30, verbose = FALSE) %>%
           FindNeighbors(reduction = "pca", dims = 1:30, verbose = FALSE) %>%
           FindClusters(resolution = 0.6, cluster.name = "clusters",
                        verbose = FALSE)

    return(obj)
})

cat("\n=== Xenium 数据摘要 ===\n")
cat("细胞数:", ncol(xenium_obj), "\n")
cat("基因数:", nrow(xenium_obj), "\n")
cat("\n聚类数:", length(unique(xenium_obj$clusters)), "\n")

# ── UMAP 聚类图 ───────────────────────────────────────────────
p_xen_umap <- DimPlot(xenium_obj, group.by = "clusters",
                       label = TRUE, pt.size = 0.2) +
              ggtitle("Xenium Unsupervised Clusters (UMAP)") +
              theme(plot.title = element_text(hjust = 0.5))
print(p_xen_umap)

# ── 空间聚类图（论文 Fig：标注前的空间结构）──────────────────
p_xen_spatial <- tryCatch({
    ImageDimPlot(xenium_obj, fov = "fov", group.by = "clusters",
                 size = 0.5, dark.background = FALSE) +
    ggtitle("Xenium Spatial — Unsupervised Clusters") +
    theme(plot.title = element_text(hjust = 0.5))
}, error = function(e) {
    cat("空间图生成失败:", e$message, "\n"); NULL
})
if (!is.null(p_xen_spatial)) print(p_xen_spatial)

ggsave(file.path(PATHS$plots$spatial, "xenium_spatial_clusters.pdf"),
       p_xen_spatial %||% p_xen_umap, width = 10, height = 9)

# 共同基因数确认
common_genes <- intersect(rownames(scrna_obj), rownames(xenium_obj))
cat("\nscRNA ∩ Xenium 共同基因数:", length(common_genes), "\n")

In [ ]:
# ============================================================
# Cell 4.5: 基因表达相关性分析（平台交叉验证）
# 论文价值：证明 Xenium 和 Flex 两个平台的表达量高度相关，
# 为后续跨模态标签转移的合理性提供基础支撑（方法章节必备图）
# ============================================================

if (file.exists(PATHS$raw$gene_panel)) {
    cat("📈 计算 Xenium vs Flex 基因表达相关性...\n")

    # ── 1. 计算各平台的基因平均表达量 ──────────────────────────
    # 只在共同基因上计算，保证维度可比
    xen_means  <- rowMeans(xenium_obj[["Xenium"]]$counts[common_genes, ])
    flex_means <- rowMeans(flex_data.obj[["RNA"]]$counts[common_genes, ] %>%
                           as.matrix())

    merged_means <- data.frame(
        gene         = common_genes,
        xenium_mean  = xen_means,
        flex_mean    = flex_means
    )

    # ── 2. 读取 gene_panel.json，标注基因来源 ──────────────────
    gene_panel  <- tryCatch(fromJSON(PATHS$raw$gene_panel), error = function(e) NULL)
    if (!is.null(gene_panel)) {
        targets     <- gene_panel$payload$targets
        panel_source <- data.frame(
            gene       = targets$source$identity$name,
            panel_type = targets$type$data$name,
            stringsAsFactors = FALSE
        )
        merged_means <- merge(merged_means, panel_source,
                              by = "gene", all.x = TRUE)
        merged_means$panel_type[is.na(merged_means$panel_type)] <- "Other"
    } else {
        merged_means$panel_type <- "Gene"
    }

    # ── 3. 相关性散点图 ─────────────────────────────────────────
    # 计算 Pearson 相关系数（log 空间）
    valid_rows <- merged_means$xenium_mean > 0 & merged_means$flex_mean > 0
    r_val <- cor(log10(merged_means$xenium_mean[valid_rows] + 1),
                 log10(merged_means$flex_mean[valid_rows]  + 1))
    cat("  Pearson r (log10 空间):", round(r_val, 4), "\n")

    p_cor <- ggplot(merged_means, aes(x = flex_mean + 1,
                                      y = xenium_mean + 1,
                                      color = panel_type)) +
        geom_point(size = 0.8, alpha = 0.7) +
        scale_colour_manual(values = c("darkcyan", "coral", "gray60")) +
        scale_x_log10() + scale_y_log10() +
        geom_abline(slope = 1, intercept = 0,
                    linetype = "dashed", color = "gray40") +
        annotate("text", x = Inf, y = -Inf,
                 label = sprintf("r = %.3f", r_val),
                 hjust = 1.1, vjust = -0.5, size = 4.5) +
        xlab("Flex scRNA Mean Expression (log10+1)") +
        ylab("Xenium Mean Expression (log10+1)") +
        ggtitle("Gene Expression Correlation: Xenium vs Flex") +
        theme_classic() +
        theme(plot.title   = element_text(hjust = 0.5),
              legend.title = element_blank())

    print(p_cor)
    ggsave(file.path(PATHS$plots$qc, "gene_expression_correlation.pdf"),
           p_cor, width = 7, height = 6)
    cat("✅ 相关性图已保存\n")

} else {
    cat("⚠️ 跳过相关性分析（找不到 gene_panel.json）\n")
    cat("  路径:", PATHS$raw$gene_panel, "\n")
}

In [ ]:
# ============================================================
# Cell 4b: Seurat 标签转移（增强版 TransferData）
#
# 改进点（对比简单版本）
# ─────────────────────────────────────────────────────────────
# 1. 高变基因取交集（优于直接用所有共同基因）
# 2. k.filter=NA（保留所有锚点），k.score=30（提高锚点质量）
# 3. l2.norm=TRUE（L2 归一化，提高稳定性）
# 4. 置信度过滤：prediction.score.max < 0.6 标记为 low_confidence
# 5. 完整可视化：UMAP + 空间图 + 小提琴图 + 分数分布直方图
#
# 输出：data/cache/xenium/seurat_label_transfer_ovarian.csv
#   列：cell_id, x, y, predicted_id, prediction_score,
#       predicted_id_filtered
# ============================================================

SEURAT_LT_FILE <- PATHS$predictions$seurat_lt

if (file.exists(SEURAT_LT_FILE) &&
    "predicted.id" %in% colnames(xenium_obj@meta.data)) {
    cat("[Cache HIT] 标签转移结果已存在\n")
    cat("预测列:", paste(grep("predicted",
                            colnames(xenium_obj@meta.data), value=TRUE),
                       collapse = ", "), "\n")

} else {
    cat("[Cache MISS] 运行 Seurat 标签转移...\n")

    # ── 1. 共同基因 ──────────────────────────────────────────────
    cat("  共同基因数:", length(common_genes), "\n")
    if (length(common_genes) < 30)
        stop("共同基因数过少（", length(common_genes), "）")

    # ── 2. Flex 子集 + 优化预处理 ────────────────────────────────
    cat("  创建 Flex 子集并优化预处理...\n")
    flex_subset <- CreateSeuratObject(
        counts = flex_data.obj[["RNA"]]$counts[common_genes, ],
        meta   = flex_data.obj@meta.data
    ) %>%
        NormalizeData(normalization.method = "LogNormalize",
                      scale.factor = 10000, verbose = FALSE) %>%
        FindVariableFeatures(nfeatures = 3000, verbose = FALSE) %>%
        ScaleData(verbose = FALSE) %>%
        RunPCA(npcs = 50, verbose = FALSE)

    # ── 3. 高变基因取交集 ────────────────────────────────────────
    # 同时在两个数据集中高变的基因，作为最有区分性的锚点特征
    flex_hv   <- VariableFeatures(flex_subset)
    xenium_hv <- VariableFeatures(xenium_obj)
    common_hv <- intersect(flex_hv, xenium_hv)

    if (length(common_hv) >= 200) {
        features_use <- common_hv[1:min(2000, length(common_hv))]
        cat("  使用高变基因交集:", length(features_use), "个\n")
    } else {
        features_use <- common_genes
        cat("  高变交集不足，使用全部共同基因:", length(features_use), "个\n")
    }

    # ── 4. 寻找锚点 ──────────────────────────────────────────────
    cat("  寻找锚点（可能需要数分钟）...\n")
    anchors <- FindTransferAnchors(
        reference           = flex_subset,
        query               = xenium_obj,
        features            = features_use,
        dims                = params$transfer_dims,
        reference.reduction = "pca",
        reduction           = "pcaproject",
        k.filter            = NA,    # 保留所有锚点（不按距离过滤）
        k.score             = 30,    # 提高锚点质量的邻居数
        verbose             = FALSE
    )
    cat("  锚点数:", nrow(anchors@anchors), "\n")

    # k.weight 自适应（锚点少时防止报错）
    kw <- min(params$transfer_k, max(5, nrow(anchors@anchors) - 1))
    cat("  k.weight:", kw, "\n")

    # ── 5. 转移标签 ──────────────────────────────────────────────
    cat("  TransferData...\n")
    label_transfer <- TransferData(
        anchorset        = anchors,
        refdata          = flex_subset$cell_type,
        dims             = params$transfer_dims,
        weight.reduction = "pcaproject",
        l2.norm          = TRUE,    # L2 归一化，提高稳定性
        k.weight         = kw
    )
    xenium_obj <- AddMetaData(xenium_obj, metadata = label_transfer)

    # ── 6. 置信度过滤 ────────────────────────────────────────────
    # 低置信度（< 0.6）的预测单独标记，不参与 GNN 比较
    if ("prediction.score.max" %in% colnames(xenium_obj@meta.data)) {
        xenium_obj@meta.data$predicted.id_filtered <-
            ifelse(xenium_obj@meta.data$prediction.score.max > params$conf_threshold,
                   as.character(xenium_obj@meta.data$predicted.id),
                   "Low Confidence")
        n_hc <- sum(xenium_obj@meta.data$prediction.score.max >
                    params$conf_threshold, na.rm = TRUE)
        cat("  高置信度细胞:", n_hc, "/", ncol(xenium_obj),
            sprintf(" (%.1f%%)\n", 100 * n_hc / ncol(xenium_obj)))
    }

    cat("✅ 标签转移完成\n")
    cat("\n预测类型分布:\n")
    print(sort(table(xenium_obj$predicted.id), decreasing = TRUE))

    # ── 7. 各细胞类型平均置信度（论文表格素材）─────────────────
    if ("prediction.score.max" %in% colnames(xenium_obj@meta.data)) {
        cat("\n各细胞类型平均置信度：\n")
        score_summary <- xenium_obj@meta.data %>%
            group_by(predicted.id) %>%
            summarise(
                n_cells     = n(),
                mean_score  = round(mean(prediction.score.max, na.rm=TRUE), 3),
                median_score= round(median(prediction.score.max, na.rm=TRUE), 3),
                .groups     = "drop"
            ) %>% arrange(desc(mean_score))
        print(score_summary)
    }

    # ── 8. 保存 CSV 供 Python notebook 对比 ──────────────────────
    coords <- tryCatch(GetTissueCoordinates(xenium_obj, scale = NULL),
                       error = function(e) NULL)
    if (!is.null(coords)) {
        cell_ids <- if ("cell" %in% colnames(coords)) coords$cell else rownames(coords)
        lt_df <- data.frame(
            cell_id          = cell_ids,
            x                = coords$x,
            y                = coords$y,
            predicted_id     = xenium_obj$predicted.id[match(cell_ids, colnames(xenium_obj))],
            prediction_score = xenium_obj$prediction.score.max[match(cell_ids, colnames(xenium_obj))],
            stringsAsFactors = FALSE
        )
        write.csv(lt_df, SEURAT_LT_FILE, row.names = FALSE)
        cat("[Saved]", SEURAT_LT_FILE, "—", nrow(lt_df), "个细胞\n")
    } else {
        cat("⚠️ 坐标提取失败，跳过 CSV 保存\n")
    }
}

# ── 9. 可视化（每次运行都执行）────────────────────────────────
cat("\n绘制标签转移结果...\n")

# UMAP：预测细胞类型
p_pred_umap <- DimPlot(xenium_obj, reduction = "umap",
                        group.by = "predicted.id",
                        label = TRUE, pt.size = 0.3, repel = TRUE) +
               ggtitle("Xenium — Predicted Cell Types (UMAP)") +
               theme(plot.title = element_text(hjust = 0.5),
                     legend.text = element_text(size = 7))
print(p_pred_umap)

# 空间图：预测细胞类型
p_pred_spatial <- tryCatch({
    ImageDimPlot(xenium_obj, fov = "fov",
                 group.by = "predicted.id",
                 size = 0.5, dark.background = FALSE) +
    ggtitle("Xenium — Predicted Cell Types (Spatial)") +
    theme(plot.title = element_text(hjust = 0.5))
}, error = function(e) { cat("空间图失败:", e$message, "\n"); NULL })
if (!is.null(p_pred_spatial)) print(p_pred_spatial)

# 小提琴图：各细胞类型的预测置信度分布（论文 Fig 必备）
if ("prediction.score.max" %in% colnames(xenium_obj@meta.data)) {
    p_violin <- ggplot(xenium_obj@meta.data,
                       aes(x = predicted.id, y = prediction.score.max,
                           fill = predicted.id)) +
        geom_violin(scale = "width", trim = TRUE, alpha = 0.8) +
        stat_summary(fun = median, geom = "point",
                     size = 1.5, color = "black") +
        geom_hline(yintercept = params$conf_threshold,
                   linetype = "dashed", color = "red", linewidth = 0.6) +
        scale_fill_viridis_d(alpha = 0.8) +
        theme_minimal() +
        theme(legend.position = "none",
              axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
              plot.title  = element_text(hjust = 0.5)) +
        ggtitle(sprintf("Prediction Score by Cell Type (threshold=%.1f)",
                        params$conf_threshold)) +
        xlab("") + ylab("Max Prediction Score") +
        ylim(0, 1)
    print(p_violin)

    # 分数整体分布直方图
    p_hist <- ggplot(xenium_obj@meta.data,
                     aes(x = prediction.score.max)) +
        geom_histogram(bins = 60, fill = "steelblue", alpha = 0.8) +
        geom_vline(xintercept = params$conf_threshold,
                   linetype = "dashed", color = "red", linewidth = 0.8) +
        theme_minimal() +
        theme(plot.title = element_text(hjust = 0.5)) +
        ggtitle("Overall Prediction Score Distribution") +
        xlab("Max Prediction Score") + ylab("Number of Cells") +
        xlim(0, 1)
    print(p_hist)

    # 保存
    ggsave(file.path(PATHS$plots$annotation, "seurat_lt_violin.pdf"),
           p_violin, width = 12, height = 6)
    ggsave(file.path(PATHS$plots$annotation, "seurat_lt_score_hist.pdf"),
           p_hist, width = 7, height = 5)
    ggsave("figures/seurat_label_transfer_spatial.pdf",
           p_pred_spatial %||% p_pred_umap, width = 10, height = 9)

    n_low <- sum(xenium_obj@meta.data$prediction.score.max <
                 params$conf_threshold, na.rm = TRUE)
    cat(sprintf("\n低置信度细胞（< %.1f）: %d (%.1f%%)\n",
                params$conf_threshold, n_low,
                100 * n_low / ncol(xenium_obj)))
}

In [ ]:
# ============================================================
# Cell 4c: 导出结果
# 输出三个文件：
#   1. cell_groups.csv     — 基础预测（cell_id / 类型 / 分数 / 一致性）
#   2. cell_predictions_full.csv — 完整分数矩阵（每个类型一列）
#   3. xenium_annotated_ovarian.rds — 完整 Seurat 对象
#
# 注意：seurat_label_transfer_ovarian.csv 已在 Cell 4b 保存
# ============================================================

if (!"predicted.id" %in% colnames(xenium_obj@meta.data)) {
    stop("请先运行 Cell 4b 完成标签转移")
}

# ── 1. 构建导出数据框 ─────────────────────────────────────────
cat("构建导出数据框...\n")
export_df <- xenium_obj@meta.data %>%
    tibble::rownames_to_column(var = "cell_id") %>%
    rename(predicted_cell_type = predicted.id)

# ── 2. 计算每个细胞的预测分数（该细胞预测类型对应的分数列）─────
score_cols <- grep("prediction\\.score\\.",
                   colnames(export_df), value = TRUE)
cat("找到预测分数列:", length(score_cols), "\n")

if (length(score_cols) > 0) {
    # 提取每个细胞对应类型的分数
    get_score <- function(row) {
        ct <- row["predicted_cell_type"]
        if (!is.na(ct) && ct %in% names(reverse_mapping)) {
            col <- paste0("prediction.score.", reverse_mapping[ct])
            if (col %in% names(row)) return(as.numeric(row[col]))
        }
        return(NA_real_)
    }
    export_df$prediction_score <- apply(export_df, 1, get_score)

    # 预测一致性（预测类型 == 最高分类型）
    get_max_type <- function(row) {
        scores  <- as.numeric(row[score_cols])
        if (all(is.na(scores))) return(NA_character_)
        max_key <- gsub("prediction\\.score\\.", "", score_cols[which.max(scores)])
        if (max_key %in% names(cell_type_mapping))
            return(cell_type_mapping[max_key])
        return(max_key)
    }
    export_df$max_score_type <- apply(export_df, 1, get_max_type)
    export_df$prediction_consistent <-
        export_df$predicted_cell_type == export_df$max_score_type

    cat("预测一致性:",
        sum(export_df$prediction_consistent, na.rm = TRUE), "/",
        nrow(export_df), "\n")
} else {
    export_df$prediction_score     <- NA
    export_df$max_score_type       <- NA
    export_df$prediction_consistent <- NA
}

# ── 3. 保存 cell_groups.csv（基础结果）───────────────────────
base_cols   <- c("cell_id", "predicted_cell_type",
                 "prediction_score", "max_score_type",
                 "prediction_consistent")
base_cols   <- base_cols[base_cols %in% colnames(export_df)]
cell_groups <- export_df %>% select(all_of(base_cols))
write.csv(cell_groups, PATHS$predictions$main, row.names = FALSE)
cat("✅ cell_groups.csv →", PATHS$predictions$main, "\n")

# ── 4. 保存完整分数矩阵 ───────────────────────────────────────
if (length(score_cols) > 0) {
    full_export <- export_df %>%
        select(cell_id, predicted_cell_type, prediction_score,
               max_score_type, prediction_consistent, all_of(score_cols))
    write.csv(full_export, PATHS$predictions$full, row.names = FALSE)
    cat("✅ 完整分数文件 →", PATHS$predictions$full, "\n")
}

# ── 5. 保存 Seurat 对象 ───────────────────────────────────────
saveRDS(xenium_obj, PATHS$predictions$seurat_obj)
cat("✅ Seurat 对象 →", PATHS$predictions$seurat_obj, "\n")

# ── 6. 生成统计报告 ───────────────────────────────────────────
dir.create(dirname(PATHS$reports$stats), recursive=TRUE, showWarnings=FALSE)
sink(PATHS$reports$stats)
cat("预测结果统计报告\n生成时间:", format(Sys.time()), "\n")
cat("总细胞数:", nrow(export_df), "\n\n")
cat("=== 细胞类型分布 ===\n")
print(sort(table(export_df$predicted_cell_type), decreasing = TRUE))
if ("prediction_score" %in% colnames(export_df)) {
    cat("\n=== 预测分数统计 ===\n")
    print(summary(export_df$prediction_score))
    cat("\n=== 各类型平均分数 ===\n")
    print(export_df %>%
          group_by(predicted_cell_type) %>%
          summarise(n=n(), mean_score=round(mean(prediction_score,na.rm=T),3),
                    .groups="drop") %>%
          arrange(desc(mean_score)))
}
sink()
cat("✅ 统计报告 →", PATHS$reports$stats, "\n")

In [ ]:
# ============================================================
# Cell 5: 完成确认
# ============================================================

cat("=== 卵巢癌 R 预处理完成 ===\n\n")
cat("缓存 / 输出文件状态：\n")
all_files <- list(
    scrna_annotated = PATHS$cache$scrna_annotated,
    xenium_base     = PATHS$cache$xenium_base,
    seurat_lt_csv   = PATHS$predictions$seurat_lt,
    cell_groups     = PATHS$predictions$main,
    full_scores     = PATHS$predictions$full,
    seurat_obj      = PATHS$predictions$seurat_obj,
    stats_report    = PATHS$reports$stats
)
for (nm in names(all_files)) {
    f    <- all_files[[nm]]
    ex   <- file.exists(f)
    size <- if (ex) paste0(round(file.size(f)/1e6, 1), " MB") else "不存在"
    cat(sprintf("  %-24s %s  (%s)\n",
                nm, if (ex) "\u2705" else "\u274c", size))
}

cat("\n下一步：运行 train_ovarian.ipynb（Python）\n")
cat("  Cell 9 会加载 seurat_label_transfer_ovarian.csv 与 GNN 结果对比\n")